# DSM050 - Disaggregating Global Wealth-Welfare Trends

## Notebook setup

In [ ]:
# Check current Python version
import sys
import warnings

EXPECTED_VERSION = (3, 13)

current_version = sys.version_info[:2]
if current_version != EXPECTED_VERSION:
    warnings.warn(
        f'Python version mismatch! Expected {EXPECTED_VERSION[0]}.{EXPECTED_VERSION[1]}, '
        f'but running on {current_version[0]}.{current_version[1]} instead.',
        UserWarning
    )

In [ ]:
# Install the required packages
%pip install -qr requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import country_converter as coco
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import seaborn as sns
import logging

from sklearn.linear_model import LinearRegression

logging.getLogger('country_converter').setLevel(logging.CRITICAL)

np.random.seed(42)

sns.set_theme(style="whitegrid")

%matplotlib inline

In [ ]:
# Create a colour palette for visualisation
# https://www.learnui.design/tools/data-color-picker.html#palette
colours = [
    "#003f5c",
    "#2e4b7f",
    "#655197",
    "#9f509d",
    "#d44e90",
    "#fa5972",
    "#ff7a49",
    "#ffa600",
]

# Plot the custom colour map
custom_cmap = mcolors.ListedColormap(colours)
gradient = np.arange(len(colours)).reshape(1, -1)
fig, ax = plt.subplots(figsize=(5, 1))
ax.imshow(gradient, aspect='auto', cmap=custom_cmap)
ax.set_axis_off()

plt.show()

## Datasets

### World Countries - GeoJSON

Available at [ArcGIS - World Countries](https://hub.arcgis.com/datasets/esri::world-countries-generalized/about).

The GeoJSON file is a reasonably clean dataset and as such contains no missing values. However the dataset utilises the ISO2 standard of country codes, while our notebook will utilise ISO3. While ISO2 is commonly used, ISO3 provides more recognisable codes. Codes are utilised to provide a standard convention that is free from ambiguity that is common when using names, such as "South Africa" and "The Republic of South Africa".

Country datasets will commonly contain duplicates. This is usually when a country is disputed, such as Somaliland breaking away from Somalia. To handle this, we utilise GeoPanda's dissolve method which merges the geometries of observations utilising the same index. While this does eliminate information contained in the dataset, it prevents ambiguity when performing the analysis.

In [ ]:
gdf_geometry = gpd.read_file("./data/World_Countries_(Generalized)_2173680399808997149.geojson")

assert gdf_geometry.isna().any(axis=1).sum() == 0, 'Country geometry source data should contain no null values.'

# Convert ISO2 to ISO3 and rename features for consistency
gdf_geometry["country_code"] = coco.convert(names=gdf_geometry["ISO"], to='ISO3')
gdf_geometry.set_index('country_code', inplace=True)
gdf_geometry = gdf_geometry[['geometry']]

print(f'Dimensions: {gdf_geometry.shape}')
print(f'Index type: {gdf_geometry.index.dtype}')
print(gdf_geometry.info())

print(f'Has duplicate indices: {gdf_geometry.index.has_duplicates}')

# Dissolve the geometries into single observations when countries are disputed. Unions the geometry.
gdf_geometry = gdf_geometry.dissolve(by=gdf_geometry.index)

print(f'Has duplicate indices after dissolving: {gdf_geometry.index.has_duplicates}')

gdf_geometry.sample(5)

## Country Income Bands

Available at [World Bank Group - Per Capita GDP](https://data.worldbank.org/indicator/NY.GDP.PCAP.CD).

The country income bands dataset contains observations for non-country entities, such as continents and regions. To handle these superfluous values, we restrict the datasat to only observations that have a matching ISO3 country code in the country geometries dataframe. This eliminates most missing values and limits the data to only entities pertinent to the analysis. However, there are still 2 observations with missing data. Since we cannot impute values, we will drop these observations.

In [ ]:
file_path = "./data/Metadata_Country_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_273495.csv"
df_income = pd.read_csv(file_path, dtype=str)

kept_columns = ['Country Code', 'IncomeGroup']
df_income = df_income[kept_columns]

df_income.rename(columns={'Country Code': 'country_code', 'IncomeGroup': 'income'}, inplace=True)

# Retain only observations that match countries in the geometries dataset
df_income = df_income[df_income['country_code'].isin(gdf_geometry.index)]
df_income.set_index('country_code', inplace=True)

# Show any observations with missing values and drop them
print(df_income[df_income.isna().any(axis=1)])
df_income.dropna(inplace=True)

# Show dataset information
print(f'Dimensions: {df_income.shape}')
print(f'Index type: {df_income.index.dtype}')
print(df_income.info())

print(f'Has duplicate indices: {df_income.index.has_duplicates}')

# Convert the income to an ordinal
income_order = ['High income', 'Upper middle income', 'Lower middle income', 'Low income']
ordinal_type = pd.CategoricalDtype(categories=income_order, ordered=True)
df_income['income'] = df_income['income'].astype(ordinal_type)

df_income.sample(5)

## Social Welfare Coverage

Here we have 2 datasets, from the International Labour Organisation and World Bank Group respectively. This is due to the sparcity of available data on social welfare coverage.

**International Labour Organisation**

Available at [ILOSTAT](https://rplumber.ilo.org/dataexplorer/?id=SDG_0131_SEX_SOC_RT_A).

This dataset contains several indicators and sources, thus we first filter the data to only those observations pertaining to the total percentage of each population covered by at least one social protection benefit.

The dataset uses country names instead of an ISO code, thus we convert the country names to ISO3 codes. This is subject to error as the conversion can make mistakes, such as converting both the "Southern Africa" region and the country "South Africa" to "ZAF". To manage this, we limit the data to the "ILO - Social Security Inquiry Database" as it contains country-based data. We then limit the dataset further to only observations matching a country geometry.

The dataset contains missing years in some countries, thus we use linear interpolation to impute missing values between two recorded years. We use this method due to the feature of interest being a continuous value that does not cease to change when unreported. An example of an instance when this is not the case would be stock prices over weekends when exchanges do not trade. In such cases we would use forward-filling of the last recorded value.


**World Bank Group**

Available at [World Bank Group - Coverage of social safety net programs](https://data.worldbank.org/indicator/per_sa_allsa.cov_pop_tot).

This dataset contains only the desired indicator albeit in an untidy format. In this case we melt the year values into "year" and "swf_pct" features. From here, we filter to known geometries and perform linear interpolation to impute missing values where possible.


**Unification**
Finally, we perform a join on the 2 datasets, using the World Bank Group data as the primary source. This is done in cases where both datasets contain values for the same year and country. In such a case, we prefer the World Bank Group data.

In [ ]:
file_path = "./data/SDG_0131_SEX_SOC_RT_A-filtered-2026-06-23.csv"
df_swf_ilo = pd.read_csv(file_path)

# The dataset contains many different labels. Filter to the percentage of the population observations only.
df_swf_ilo = df_swf_ilo[df_swf_ilo["classif1.label"] == "Function: Population covered by at least one social protection benefit"]

# The dataset breaks down data by sex. We only want the totals.
df_swf_ilo = df_swf_ilo[df_swf_ilo["sex.label"] == "Total"]

# The dataset uses multiple sources. Limit to ILO - Social Security Inquiry Database as it contains country data
df_swf_ilo = df_swf_ilo[df_swf_ilo["source.label"] == "ILO - Social Security Inquiry Database"]

# The dataset uses country names while others use ISO2 or ISO3. Convert to ISO3.
df_swf_ilo["country_code"] = coco.convert(df_swf_ilo['ref_area.label'], to='ISO3')

# Keep only the features of interest and rename them for consistency.
kept_columns = ["time", "obs_value", "country_code"]
df_swf_ilo = df_swf_ilo[kept_columns]
df_swf_ilo.rename(columns={'time': 'year', 'obs_value': 'swf_pct'}, inplace=True)

# Convert the remaining features into appropriate types
df_swf_ilo["year"] = pd.to_datetime(df_swf_ilo["year"], format="%Y")
df_swf_ilo["country_code"] = df_swf_ilo["country_code"].astype(str)

print(f'Minimum year: {df_swf_ilo["year"].min()}')
print(f'Maximum year: {df_swf_ilo["year"].max()}')

# Filter to only entries that exist in the country dataset
df_swf_ilo = df_swf_ilo[df_swf_ilo['country_code'].isin(gdf_geometry.index)]

# Check how many countries are missing.
countries_contained = df_swf_ilo['country_code'].unique()
if len(countries_contained) != len(gdf_geometry.index):
    print(f"There are {len(gdf_geometry.index) - len(countries_contained)} unrepresented countries")
    missing_countries = set(gdf_geometry.index) - set(countries_contained)
    print(coco.convert(missing_countries, to="name_short"))

print(f'Mean entries per country: {df_swf_ilo['country_code'].value_counts().mean()}')

# Set indices and sort
df_swf_ilo.set_index(['country_code', 'year'], inplace=True)
df_swf_ilo.sort_index(inplace=True)

# Interpolate any missing values between entries per country
df_swf_ilo['swf_pct'] = df_swf_ilo.groupby('country_code', group_keys=False)['swf_pct'].apply(lambda x: x.interpolate(method='linear'))
print(f'Mean entries per country after imputation: {df_swf_ilo.reset_index()['country_code'].value_counts().mean()}')

print(f'Has duplicate indices: {df_swf_ilo.index.has_duplicates}')
print(df_swf_ilo.info())

df_swf_ilo.head()

In [ ]:
file_path = "./data/API_PER_SA_ALLSA.COV_POP_TOT_DS2_en_csv_v2_329606.csv"
df_swf_wbg = pd.read_csv(file_path, skiprows=4)

# Determine the features to keep and drop the others
years = pd.date_range('1960', '2025', freq='YS').year.astype(str).tolist()
kept_columns = ['Country Code'] + years
df_swf_wbg = df_swf_wbg[kept_columns]

# Rename for consistency
df_swf_wbg.rename(columns={'Country Code': 'country_code'}, inplace=True)

# Melt the year features by country into a single p.c GDP feature.
df_swf_wbg = df_swf_wbg.melt(
    id_vars=['country_code'],
    var_name='year',
    value_name='swf_pct'
)

# Convert the remaining features into appropriate types
df_swf_wbg["year"] = pd.to_datetime(df_swf_wbg["year"], format="%Y")
df_swf_wbg["country_code"] = df_swf_wbg["country_code"].astype(str)

print(f'Minimum year: {df_swf_wbg["year"].min()}')
print(f'Maximum year: {df_swf_wbg["year"].max()}')

# Check how many countries are missing.
countries_contained = df_swf_wbg['country_code'].unique()
if len(countries_contained) != len(gdf_geometry.index):
    print(f"There are {len(gdf_geometry.index) - len(countries_contained)} unrepresented countries")
    missing_countries = set(gdf_geometry.index) - set(countries_contained)
    print(coco.convert(missing_countries, to="name_short"))

print(f'Mean entries per country: {df_swf_wbg['country_code'].value_counts().mean()}')

# Limit observations to countries for which we have geometry.
df_swf_wbg = df_swf_wbg[df_swf_wbg['country_code'].isin(gdf_geometry.index)]

# Set the indices and sort them
df_swf_wbg.set_index(['country_code', 'year'], inplace=True)
df_swf_wbg.sort_index(inplace=True)

# Impute missing values chronologically per country
df_swf_wbg['swf_pct'] = df_swf_wbg.groupby('country_code', group_keys=False)['swf_pct'].apply(lambda x: x.interpolate(method='linear'))
print(f'Mean entries per country after imputation: {df_swf_wbg.reset_index()['country_code'].value_counts().mean()}')

print(f'Has duplicate indices: {df_swf_wbg.index.has_duplicates}')
print(df_swf_wbg.info())

df_swf_wbg.head()

In [ ]:
# Unify the SWF datasets using df_swf_wbg as the primary source
df_swf_unified = pd.merge(df_swf_ilo, df_swf_wbg, how='outer', left_index=True, right_index=True)
df_swf_unified['swf_pct'] = df_swf_unified['swf_pct_y'].combine_first(df_swf_unified['swf_pct_x'])
df_swf_unified = df_swf_unified.drop(columns=['swf_pct_x', 'swf_pct_y'])
df_swf_unified.sample(5)

## Per Capita GDP

Available at [Wold Bank Group - GDP per capita](https://data.worldbank.org/indicator/NY.GDP.PCAP.CD).

The dataset contains the PC GDP as a series of year features. THus we melt the years features into "year" and "pc_gdp" features. Again, we limit the observations to those that match our coutnry geometries. Finally we use linear interpolation to impute missing values where possible.

In [ ]:
df_gdp = pd.read_csv("./data/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_273495.csv", skiprows=4)

# Determine the features to keep and drop the others
years = pd.date_range('1960', '2025', freq='YS').year.astype(str).tolist()
kept_columns = ['Country Code'] + years
df_gdp = df_gdp[kept_columns]

# Rename for consistency
df_gdp.rename(columns={'Country Code': 'country_code'}, inplace=True)

# Melt the year features by country into a single p.c GDP feature.
df_gdp = df_gdp.melt(
    id_vars=['country_code'],
    var_name='year',
    value_name='pc_gdp'
)

# Convert the remaining features into appropriate types
df_gdp["year"] = pd.to_datetime(df_gdp["year"], format="%Y")
df_gdp["country_code"] = df_gdp["country_code"].astype(str)

print(f'Minimum year: {df_gdp["year"].min()}')
print(f'Maximum year: {df_gdp["year"].max()}')

# Check how many countries are missing.
countries_contained = df_gdp['country_code'].unique()
if len(countries_contained) != len(gdf_geometry.index):
    print(f"There are {len(gdf_geometry.index) - len(countries_contained)} unrepresented countries")
    missing_countries = set(gdf_geometry.index) - set(countries_contained)
    print(coco.convert(missing_countries, to="name_short"))

print(f'Mean entries per country: {df_gdp['country_code'].value_counts().mean()}')

# Limit observations to countries for which we have geometry.
df_gdp = df_gdp[df_gdp['country_code'].isin(gdf_geometry.index)]

# Set the indices and sort them
df_gdp.set_index(['country_code', 'year'], inplace=True)
df_gdp.sort_index(inplace=True)

# Impute missing values chronologically per country
df_gdp['pc_gdp'] = df_gdp.groupby('country_code', group_keys=False)['pc_gdp'].apply(lambda x: x.interpolate(method='linear'))
print(f'Mean entries per country after imputation: {df_gdp.reset_index()['country_code'].value_counts().mean()}')

print(f'Has duplicate indices: {df_gdp.index.has_duplicates}')
print(df_gdp.info())

df_gdp.sample(5)

## Country Population

Available at [World Bank Group - Population](https://data.worldbank.org/indicator/SP.POP.TOTL).

This dataset contains the coutnry population spread accross multiple year features. Thus we melt the multiple features into "year" and "population" features. We then filter the dataset to only include obersvations for which we have geometries. Then we use linear interpolation to impute missing values where possible.

In [ ]:
df_population = pd.read_csv("./data/API_SP.POP.TOTL_DS2_EN_csv_v2_327505.csv", skiprows=4)

# Determine features to keep
years = pd.date_range('1960', '2025', freq='YS').year.astype(str).tolist()
kept_columns = ['Country Code'] + years
df_population = df_population[kept_columns]

# Rename features for consistency
df_population.rename(columns={'Country Code': 'country_code'}, inplace=True)

# Melt the year features by country into a single population feature.
df_population = df_population.melt(
    id_vars=['country_code'],
    var_name='year',
    value_name='population'
)

# Convert the remaining features into appropriate types
df_population["year"] = pd.to_datetime(df_population["year"], format="%Y")
df_population["country_code"] = df_population["country_code"].astype(str)

print(f'Minimum year: {df_population["year"].min()}')
print(f'Maximum year: {df_population["year"].max()}')

# Check how many countries are missing.
countries_contained = df_population['country_code'].unique()
if len(countries_contained) != len(gdf_geometry.index):
    print(f"There are {len(gdf_geometry.index) - len(countries_contained)} unrepresented countries")
    missing_countries = set(gdf_geometry.index) - set(countries_contained)
    print(coco.convert(missing_countries, to="name_short"))

print(f'Mean entries per country: {df_population['country_code'].value_counts().mean()}')

# Limit observations to countries for which we have geometry.
df_population = df_population[df_population['country_code'].isin(gdf_geometry.index)]

# Set the indices and sort them
df_population.set_index(['country_code', 'year'], inplace=True)
df_population.sort_index(inplace=True)

# Impute missing values chronologically per country
df_population['population'] = df_population.groupby('country_code', group_keys=False)['population'].apply(lambda x: x.interpolate(method='linear'))
print(f'Mean entries per country after imputation: {df_population.reset_index()['country_code'].value_counts().mean()}')

print(f'Has duplicate indices: {df_population.index.has_duplicates}')
print(df_population.info())

df_population.sample(5)

## Exploratory Data Analysis

In [ ]:
# Merge gdp, population, and swf using compound key
df_unified = pd.merge(df_gdp, df_population, how='outer', left_index=True, right_index=True)
df_unified = pd.merge(df_unified, df_swf_unified, how='outer', left_index=True, right_index=True)

# Remove the multi-index to join the geometries using only the country_code
df_unified = df_unified.reset_index()
df_unified = df_unified.merge(gdf_geometry, left_on="country_code", right_index=True, how="left")

# Recreate the multi-index and validate it for duplicates
df_unified.set_index(['country_code', 'year'], inplace=True)
print(f'Has duplicate indices: {df_unified.index.has_duplicates}')

df_unified.sample(5)

In [ ]:
df_unified.reset_index().describe()

In [ ]:
# The data will be highly skewed so we tranform to log values
transformed_data = np.log1p(df_unified['population'].dropna())
median_log = transformed_data.median()
mean_log = transformed_data.mean()

plt.figure(figsize=(10, 6))
n, bins, patches = plt.hist(
    transformed_data, 
    bins=25, 
    edgecolor='white', 
    color=colours[0], 
    alpha=0.9, 
    density=True
)

# Markers for mean and median
plt.axvline(median_log, color=colours[7], linestyle='--', linewidth=2, 
            label=f'Median: {median_log:.2f} (≈ {int(np.expm1(median_log)):,})')
plt.axvline(mean_log, color=colours[4], linestyle=':', linewidth=2, 
            label=f'Mean: {mean_log:.2f} (≈ {int(np.expm1(mean_log)):,})')

# Labels
plt.title('Global Distribution: Population (1960-2025)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel(r'Log-Transformed Population [ $\ln(1 + x)$ ]', fontsize=11, labelpad=10)
plt.ylabel('Probability Density', fontsize=11, labelpad=10)

# Transform x-ticks to monetary values for readability
tick_positions = np.linspace(transformed_data.min(), transformed_data.max(), 6)
# Convert back from log scale: e^ticks - 1
tick_labels = [f"{int(np.expm1(pos)):,}" for pos in tick_positions]
plt.xticks(tick_positions, tick_labels)

plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.legend(frameon=True, facecolor='white', edgecolor='none')
plt.tight_layout()

plt.show()

In [ ]:
# The data will be highly skewed so we tranform to log values
transformed_data = np.log1p(df_unified['pc_gdp'].dropna())
median_log = transformed_data.median()
mean_log = transformed_data.mean()

plt.figure(figsize=(10, 6))
n, bins, patches = plt.hist(
    transformed_data, 
    bins=25, 
    edgecolor='white', 
    color=colours[0], 
    alpha=0.9, 
    density=True
)

# Markers for mean and median
plt.axvline(median_log, color=colours[7], linestyle='--', linewidth=2, 
            label=f'Median: {median_log:.2f} (≈ ${int(np.expm1(median_log)):,})')
plt.axvline(mean_log, color=colours[4], linestyle=':', linewidth=2, 
            label=f'Mean: {mean_log:.2f} (≈ ${int(np.expm1(mean_log)):,})')

# Labels
plt.title('Global Distribution: GDP Per Capita (1960–2025)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel(r'Log-Transformed GDP per Capita [ $\ln(1 + x)$ ]', fontsize=11, labelpad=10)
plt.ylabel('Probability Density', fontsize=11, labelpad=10)

# Transform x-ticks to monetary values for readability
tick_positions = np.linspace(transformed_data.min(), transformed_data.max(), 6)
# Convert back from log scale: e^ticks - 1
tick_labels = [f"${int(np.expm1(pos)):,}" for pos in tick_positions]
plt.xticks(tick_positions, tick_labels)

plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.legend(frameon=True, facecolor='white', edgecolor='none')
plt.tight_layout()

plt.show()

In [ ]:
swf_data = df_unified['swf_pct'].dropna()
median_swf = swf_data.median()
mean_swf = swf_data.mean()

plt.figure(figsize=(10, 6))
n, bins, patches = plt.hist(
    swf_data, 
    bins=20,
    range=(0, 100),
    edgecolor='white', 
    color=colours[0], 
    alpha=0.9, 
    density=True
)

# Add markers for mean and median
plt.axvline(median_swf, color=colours[7], linestyle='--', linewidth=2, 
            label=f'Median: {median_swf:.1f}%')
plt.axvline(mean_swf, color=colours[4], linestyle=':', linewidth=2, 
            label=f'Mean: {mean_swf:.1f}%')

# Labels
plt.title('Global Distribution: Social Welfare Coverage (1960-2025)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Social Welfare Coverage (% of Population)', fontsize=11, labelpad=10)
plt.ylabel('Probability Density', fontsize=11, labelpad=10)

# X-ticks to represent the percentages
tick_positions = np.linspace(0, 100, 6)
tick_labels = [f"{int(pos)}%" for pos in tick_positions]
plt.xticks(tick_positions, tick_labels)

plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.legend(frameon=True, facecolor='white', edgecolor='none')
plt.tight_layout()

plt.show()

In [ ]:
df_unified.info()

In [ ]:
total_countries = df_unified.index.get_level_values(0).nunique()

# Count non-null values per year and set the index to "year" only
yearly_counts = df_unified[['pc_gdp', 'population', 'swf_pct']].notna().groupby(level=1).sum()
yearly_counts.index = yearly_counts.index.year

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 8), sharex=True)

features = ['pc_gdp', 'population', 'swf_pct']
plot_colours = [custom_cmap(0.0), custom_cmap(0.5), custom_cmap(1.0)]

for ax, feature, color in zip(axes, features, plot_colours):
    # Plot the line
    ax.plot(yearly_counts.index, yearly_counts[feature], color=color, linewidth=2)
    # Fill underneath to make data density gaps immediately obvious
    ax.fill_between(yearly_counts.index, yearly_counts[feature], color=color, alpha=0.2)

    ax.set_ylim(0, total_countries)
    
    # Labels and formatting
    ax.set_ylabel('Non-Null Count', fontsize=10)
    ax.set_title(f'Data Availability Over Time: {feature}', fontsize=12, loc='left')
    ax.grid(True, linestyle='--', alpha=0.5)

# Final layout adjustments
axes[-1].set_xlabel('Year', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
corr_data = df_unified.drop(columns='geometry').corr(method="pearson")
corr_mask = np.triu(np.ones_like(corr_data, dtype=bool))
sns.heatmap(corr_data, mask=corr_mask, annot=True, cmap='coolwarm')
plt.show()

In [ ]:
df_2010 = df_unified.drop(columns='geometry').xs('2010-01-01', level=1).copy()
df_2010['log_pc_gdp'] = np.log1p(df_2010['pc_gdp'])
df_2010['log_population'] = np.log1p(df_2010['population'])

plot_cols = ['log_population', 'log_pc_gdp', 'swf_pct']
g = sns.pairplot(
    data=df_2010[plot_cols].dropna(),
    diag_kind='kde',                    # Smooth distribution curves are cleaner than bars
    plot_kws={
        'alpha': 0.7,                   # Transparency helps see overlapping points
        'edgecolor': 'white',           # Sharp white outline for clarity
        's': 55                         # Consistent point size
    },
    height=3, 
    aspect=1.1
)

# 5. Fine-tune axis labels and titles for readability
# Map clean math symbols to the log-transformed axes
labels = {
    'Log pc_gdp': r'Log GDP per Capita' + '\n' + r'[ $\ln(1+x)$ ]',
    'Log population': r'Log Population' + '\n' + r'[ $\ln(1+x)$ ]',
    'swf_pct': 'SWF Coverage\n(% of Population)'
}

for ax in g.axes.flat:
    if ax is not None:
        # Check and update X labels
        xlabel = ax.get_xlabel()
        if xlabel in labels:
            ax.set_xlabel(labels[xlabel], fontsize=10, fontweight='bold', labelpad=10)
        # Check and update Y labels
        ylabel = ax.get_ylabel()
        if ylabel in labels:
            ax.set_ylabel(labels[ylabel], fontsize=10, fontweight='bold', labelpad=10)

# 6. Add an overarching dashboard title
g.fig.suptitle('Multivariate Pairwise Profile (Benchmark Year: 2010)', fontsize=14, fontweight='bold', y=1.03)

plt.show()

NB: This is a key finding. At the global level, population is uncorrelated. At the local level it is very correlated. This is a clear-cut example of Simpson's paradox.

In [ ]:
sns.pairplot(df_unified.loc['ZAF'].drop(columns='geometry'))

In [ ]:
n = 3  # A correlation requires at least 3 data points though this is still very little.

country_corr = (
    df_unified.loc[:, ["pc_gdp", "population", "swf_pct"]].dropna().groupby(level='country_code')
    .filter(lambda x: len(x) >= n)
    .groupby(level='country_code')
    .corr(method="pearson")
)

country_corr.head()

In [ ]:
df_corr = country_corr.xs("population", level=1)[["pc_gdp", "swf_pct"]]
df_corr.rename(columns={"pc_gdp": "corr_pop_gdp", "swf_pct": "corr_pop_swf"}, inplace=True)
df_corr.sample(5)

In [ ]:
def assign_country_type(row):
    gdp = row["corr_pop_gdp"]
    swf = row["corr_pop_swf"]
    
    # Mark any row with missing values as UNK
    if pd.isna(gdp) or pd.isna(swf):
        return "UNK"
        
    # Classify observations
    if gdp > 0 and swf > 0:
        return "A-1"
    elif gdp > 0 and swf <= 0:
        return "A-2"
    elif gdp <= 0 and swf > 0:
        return "B-1"
    else:  # gdp <= 0 and swf <= 0
        return "B-2"
    
df_corr["class_type"] = df_corr.apply(assign_country_type, axis=1)

print(df_corr["class_type"].value_counts(dropna=False))

df_corr.head()

In [ ]:
df_map = gdf_geometry.join(df_corr, on='country_code', how='left')
df_map['class_type'] = df_map['class_type'].fillna('UNK')
df_map['corr_pop_gdp'] = df_map['corr_pop_gdp'].fillna(0)
df_map['corr_pop_swf'] = df_map['corr_pop_swf'].fillna(0)
df_map.head()

In [ ]:
# Create colour palette
plot_colors = [custom_cmap(0.0), custom_cmap(0.3), custom_cmap(0.6), custom_cmap(1.0)]
color_map = {
    "A-1": plot_colors[0],
    "A-2": plot_colors[1],
    "B-1": plot_colors[2],
    "B-2": plot_colors[3],
    "UNK": "#d3d3d3"
}

# Dynamically shift the colour based on correlation
def get_dynamic_color(row):
    cls = row["class_type"]
    
    # Handle UNK or NaN
    if cls == "UNK" or pd.isna(row["corr_pop_gdp"]) or pd.isna(row["corr_pop_swf"]):
        return mcolors.to_rgba(color_map['UNK'])
    
    # Compute Euclidean distance and normalise
    gdp = row["corr_pop_gdp"]
    swf = row["corr_pop_swf"]
    distance = np.sqrt(gdp**2 + swf**2)
    intensity = min(distance / np.sqrt(2), 1.0)
    
    # Convert base hex colour to RGB, then to HSL
    base_rgb = mcolors.to_rgb(color_map[cls])
    h, l, s = mcolors.rgb_to_hsv(base_rgb)
    
    # Strong correlation (intensity=1) -> Original coloor
    # Weak correlation (intensity=0) -> Lightened colour
    mix_factor = 0.2 + (0.8 * intensity)  # Ensure that we don't lighten all the way to white
    dynamic_rgb = tuple(1.0 - (1.0 - c) * mix_factor for c in base_rgb)
    
    return mcolors.to_rgb(dynamic_rgb)

# Compute a colour for each observation
df_map["dynamic_color"] = df_map.apply(get_dynamic_color, axis=1)

fig, ax = plt.subplots(1, 1, figsize=(15, 10))

# Plot the choropleth
df_map.plot(
    ax=ax,
    color=df_map["dynamic_color"], 
    edgecolor="#333333",             
    linewidth=0.4                  
)

# 6. Build the Legend (Displays the pure base colors for reference)
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label)
    for label, color in color_map.items()
]
# Add a descriptor item to explain the shading variation to readers
legend_elements.append(plt.Line2D([0], [0], marker='', color='w', label='Lighter shade indicate weak correlation'))

ax.legend(handles=legend_elements, title="Country Classification", loc="lower left")
ax.set_title("Global Country Classification", fontsize=14)
ax.set_axis_off()

plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))

# Join the country income bands
df_matrix = df_map[df_map["class_type"] != "UNK"].dropna(subset=["corr_pop_gdp", "corr_pop_swf"])
df_matrix = df_matrix.join(df_income, on="country_code", how="left")

# Scatter Plot
sns.scatterplot(
    data=df_matrix,
    x="corr_pop_swf",
    y="corr_pop_gdp",
    hue="class_type",  # Colour based on country classification
    hue_order=["A-1", "A-2", "B-1", "B-2"],
    palette=color_map,
    size="income",  # Size based on income  band
    sizes=(40, 200),
    alpha=0.7,
    ax=ax
)

# Quadrant crosshair axes lines at (0,0)
ax.axhline(0, color="black", linestyle="--", linewidth=1.2, alpha=0.6)
ax.axvline(0, color="black", linestyle="--", linewidth=1.2, alpha=0.6)

# Country labels text loop
for idx, row in df_matrix.iterrows():
    ax.text(
        row["corr_pop_swf"] + 0.015, 
        row["corr_pop_gdp"] + 0.015, 
        str(idx), 
        fontsize=9, 
        alpha=0.8,
        verticalalignment='bottom'
    )

ax.text(0.5, 0.5, "A-1\n(+, +)", fontsize=16, color=color_map['A-1'], alpha=0.3, weight="bold", ha="center")
ax.text(-0.5, 0.5, "A-2\n(-, +)", fontsize=16, color=color_map['A-2'], alpha=0.3, weight="bold", ha="center")
ax.text(0.5, -0.5, "B-1\n(+, -)", fontsize=16, color=color_map['B-1'], alpha=0.3, weight="bold", ha="center")
ax.text(-0.5, -0.5, "B-2\n(-, -)", fontsize=16, color=color_map['B-2'], alpha=0.3, weight="bold", ha="center")

# Labels and Limits
ax.set_title("Country Matrix", fontsize=14, pad=15)
ax.set_ylabel("Population vs GDP Correlation (corr_pop_gdp)", fontsize=11)
ax.set_xlabel("Population vs SWF Correlation (corr_pop_swf)", fontsize=11)

ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
ax.legend(
    title="Classification", 
    bbox_to_anchor=(1, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


In [ ]:
# Create a copy of the observations without the multi-index
df_trajectories = df_unified.copy().reset_index()

fig, ax = plt.subplots(figsize=(12, 7))

sns.scatterplot(data=df_trajectories, x='pc_gdp', y='swf_pct', color='#cbd5e1', alpha=0.3, s=15, label='Global Country-Years', ax=ax)

# Compute the regression line of log(GDP)
df_clean = df_unified[['pc_gdp', 'swf_pct']].dropna()
X_log = np.log10(df_clean[['pc_gdp']].values)
y = df_clean['swf_pct'].values
model = LinearRegression().fit(X_log, y)

# Generate points along the regression line
x_span = np.linspace(df_trajectories['pc_gdp'].min(), df_trajectories['pc_gdp'].max(), 500)
x_span_log = np.log10(x_span.reshape(-1, 1))
y_pred = model.predict(x_span_log)

# Plot the regression line to represent the global correlation
ax.plot(x_span, y_pred, color='#0f172a', linewidth=3, label="Global Baseline Trend (r = 0.58)", zorder=3)

# Setup countries to plot the trajectories of and their colours
focus_countries = {
    'GHA': (color_map['A-1'], 'Ghana (Nordic)'),
    'TUR': (color_map['A-2'], 'Turkey (Market)'),
    'ZAF': (color_map['B-1'], 'South Africa (Clientelist)'),
    'AFG': (color_map['B-2'], 'Afganistan (Collapse)')
}

# For each country, plot the trajectory of GDP/SWF
for code, (color, label) in focus_countries.items():
    country_data = df_trajectories[df_trajectories['country_code'] == code]
    if not country_data.empty:
        sns.scatterplot(data=country_data, x='pc_gdp', y='swf_pct', color=color, s=45, zorder=5, ax=ax)
        ax.plot(country_data['pc_gdp'], country_data['swf_pct'], color=color, linewidth=2.5, zorder=4, label=label)
        ax.text(country_data['pc_gdp'].iloc[0], country_data['swf_pct'].iloc[0], f" {country_data['year'].iloc[0]}", fontsize=9, color=color, zorder=6)
        ax.text(country_data['pc_gdp'].iloc[-1], country_data['swf_pct'].iloc[-1], f" {country_data['year'].iloc[-1]}", fontsize=9, color=color, weight='bold', zorder=6)

# Apply log scaling to ensure the regression line is presented as a straight line and not a curve.
ax.set_xscale('log')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"${int(x):,}"))

# Construct the title
clean_title = "The Aggregation Illusion: Global Wealth-Welfare Alignment vs. Local Trajectories"
ax.set_title(clean_title, fontsize=13, weight='bold', loc='left', pad=20, color='#1e293b')

# Set legend and axes
ax.set_xlabel("Per Capita GDP (Logarithmic Scale)", fontsize=11, labelpad=10)
ax.set_ylabel("Social Welfare Expenditure (%)", fontsize=11, labelpad=10)
ax.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='none')

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()